# Programmieraufgabe 4: QR-Algorithmus für reelle Matrizen

**Abgabe in den Programmiertutorien am 2./3. Juli 2025.**

In diesem Notebook erweitern wir den QR-Algorithmus aus Programmieraufgabe 3 so, dass er auch mit reellen Matrizen mit komplex konjugierten Eigenwerten zurecht kommt und dabei nur reelle Operationen benutzt. Dazu verwenden wir die "bulge chasing" Technik.

Die Aufgabe erscheint auf den ersten Blick vielleicht etwas lang. Das liegt aber nur daran, dass die zu implementierende Vorgehensweise nochmal im Detail erklärt wird und einige Code-Bausteine, insbesondere zum Testen von Prozeduren, vorgebeben sind.  

In der Programmieraufgabe werden Sie die Prozeduren `hess` und `qr_alg_shift` aus den Teilen (e) bzw. (h) des Notebooks aus Programmieraufgabe 3 wiederverwenden können. Sie können sich diese beiden Prozeduren entweder aus Ihrer eigenen Lösung oder aus dem im Ilias bereit gestellten Lösungsvorschlag (ab 24. Juni verfügbar) an geeigneter Stelle kopieren.

Wir benötigten in diesem Notebook die folgenden Module:

In [1]:
import numpy as np
import scipy.linalg as spla  # für Matrixzerlegungen und co
import numpy.random as rnd   # für alles, was mit Zufallszahlen zu tun hat

Außerdem können wieder die beiden Prozeduren `printvector` und `printmatrix` zur Ausgabe von Vektoren und Matrizen verwendet werden.

In [2]:
def printvector(v):
    if v.dtype == "int":
        print(''.join([' {:4}'.format(item) for item in v])+"\n")
    elif v.dtype == "complex128":
        print(''.join([' {:16.3f}'.format(item) for item in v])+"\n")
    else:
        print(''.join([' {:7.3f}'.format(item) for item in v])+"\n")

In [3]:
def printmatrix(A):
    if A.dtype == "int":
        print('\n'.join([''.join(['  {:4}'.format(item) for item in row]) for row in A])+"\n")
    elif A.dtype == "complex128":
        print('\n'.join([''.join(['  {:16.3f}'.format(item) for item in row]) for row in A])+"\n")   
    else:
        print('\n'.join([''.join(['  {:7.3f}'.format(item) for item in row]) for row in A])+"\n")       

# Problemstellung & Modellmatrix

Wir betrachten als Modellmatrix zunächst die Matrix $A = A_6$ aus dem Notebook zu Programmieraufgabe 3. Diese wurde durch eine Ähnlichkeitstransformation aus der Blockdiagonalmatrix 
$$
D_6 = \begin{pmatrix} 2 \\ & 1 \\ && 1 & -1 \\ && 1 & 1 \\ &&&& \frac12 \end{pmatrix}
$$
erzeugt, d.h. $A = S^{-1} D_6 S$ für eine invertierbare Matrix $S$. Der $2\times2$ Diagonalblock hat das komplex konjugierte Eigenwertpaar $1\pm\mathrm{i}$. Dasselbe gilt dann offensichtlich auch für die Matrizen $D_6$ bzw. $A$.

Zunächst erzeugen wir die Matrix $A$ wie im letzten Notebook und bringen sie mit der Prozedur `hess` in Hessenberg-Form:

<!-- Am Ende des Notebooks wollen wir eine auf dem QR-Algorithmus basierende Prozedur haben, die auch von dieser Matrix $A$ alle Eigenwerte berechnen kann, und dabei alle eigentlichen Iterationen in reeller Arithmetik durchführt. -->


In [4]:
def hess(A):
    n = np.size(A,0) # Anzahl Zeilen/Spalten
    for j in range(n-2):
        # Spiegele erste Spalte von A[j+1:,j:] auf alpha-faches von e_1
        
        # Baue passenden Householder-Vektor zusammen
        x = np.copy(A[j+1:,j]) # Erste Spalte von A[j+1:,j:]
        alpha = - x[0]/np.abs(x[0]) * np.linalg.norm(x)
        v = x
        v[0] -= alpha
        v = v/np.linalg.norm(v)
        
        # Wende Q = I - 2vv^H von links an
        A[j+1,j] = alpha
        A[j+2:,j] = 0
        A[j+1:,j+1:] += np.outer( (-2*v) , v.conj() @ A[j+1:,j+1:] )

        # Wende Q = I - 2vv^H von rechts an
        A[:,j+1:] -= np.outer( A[:,j+1:] @ (2*v) , v.conj() )

    return A

In [5]:
# (Block-)Diagonalmatrix
A = np.array([[2,0,0,0,0],[0,1,0,0,0],[0,0,1,-1,0],[0,0,1,1,0],[0,0,0,0,1/2]])

# Ähnlichkeitstransformation
S = np.array([
    [2, -1, 1, 0, -1],
    [-1, 1, 2, 2, 2],
    [-1, 0, 2, -1, 1],
    [-1, 2, 2, 2, 1],
    [2, -1, 2, 0, -1]
])
S_inv = spla.inv(S)

# Finale Matrix A
A = S_inv @ A @ S

# Matrix A in Hessenberg-Form
A_hess = hess(A.copy())

printmatrix(A_hess)

   16.500   11.093   -2.337   10.916    0.317
  -20.700  -13.821    2.699  -15.152    0.446
    0.000    0.063    1.121   -1.915   -0.454
    0.000    0.000    0.626    0.711   -1.337
    0.000    0.000    0.000   -0.205    0.989



Wir arbeiten uns schrittweise in Richtung des finalen Programms vor:

# Schritt 1: Bulge chasing
Gegeben sei hier zunächst eine Matrix $A\in\mathbb{R}^{n\times n}$ mit der Struktur
$$
A = \begin{pmatrix}
    * & * & * & * & \cdots & * \\
    * & * & * & * & & \vdots \\
    \otimes & * & * & * & & \\
    \otimes & \otimes & * & * & & \\
     & & & \ddots & \ddots & \\
     & & & & * & *
    \end{pmatrix}.
$$
Die drei mit $\otimes$ markierten Einträge bilden den _bulge_. Wenn dieser bulge nicht wäre, dann hätte die Matrix Hessenberg-Form.

**(a) Ändern Sie die Prozedur `hess` aus Programmieraufgabe 3 zu einer Prozedur `bulge_chasing` ab, die eine Matrix $A$ mit der obigen Struktur durch Householder-Transformationen auf Hessenberg-Form bringt (siehe Schritt 4 aus dem Francis' QR-Schritt). Achten Sie dabei darauf, in jedem Schritt nur die Zeilen/Spalten der Matrix zu bearbeiten, bei denen dies nötig ist.**

In [6]:
def bulge_chasing(A):
    n = np.size(A,0) # Anzahl Zeilen/Spalten
    for j in range(n-2):
        # Ein Vektor welcher Länge muss auf ein Vielfaches von e_1 transformiert werden?
        if j < n-3: # Spalten 0,...,n-4
            m = 3
        else: # Spalte n-3 (drittletzte Spalte)
            m = 2
            
        ### Spiegele die ersten m Einträge unterhalb der Diagonalen in der j-ten Spalte von A auf Vielfaches von e_1
        
        # Baue passenden Householder-Vektor zusammen
        x = np.copy( A[j+1:j+m+1, j] ) # Zu spiegelnder Vektor (mit drei/zwei Einträgen)
        alpha = - np.sign(x[0]) * np.linalg.norm(x)
        v = x
        v[0] -= alpha
        v = v/np.linalg.norm(v)
        
        # Wende Q = I - 2vv^T von links an --> drei/zwei Zeilen von A bearbeiten (und zwar nur die Spalten j,...,n-1)
        A[j+1, j] = alpha
        A[j+2:j+m+1, j] = 0
        A[j+1:j+m+1, j+1:] += np.outer( (-2*v) , v @ A[j+1:j+m+1, j+1:] )

        # Wende Q = I - 2vv^T von rechts an --> drei/zwei Spalten von A bearbeiten (und zwar nur die Zeilen 0,...,j+4)
        A[:j+5,j+1:j+m+1] -= np.outer( A[:j+5,j+1:j+m+1] @ (2*v) , v )

    return A

Änderungen gegenüber `hess`:
- Nur zwei Einträge (bzw. in der drittletzten Spalte ein Eintrag) müssen in jeder Spalte eliminiert werden.
- Die zu spiegelnden Vektoren haben also drei (bzw. zwei) Einträge, genauso wie die Householder-Vektoren der dazu nötigen Householder-Transformationen $Q$.
- Bei der Multiplikation mit $Q$ von links oder von rechts müssen nur drei (bzw. zwei) Zeilen oder Spalten bearbeitet werden.
- Bei der Multiplikation von rechts kann man außerdem noch ausnutzen, in welchen Zeilen denn überhaupt Nicht-Null-Einträge vorkommen.
- Die reelle Variante der HH-Trafos wird verwendet (die komplexe Variante wäre nicht falsch, da sie bei reellen Einträgen der reellen Variante entspricht).

Sie können die Prozedur mit folgemdem Code testen. Im ersten Code-Block wird eine zufällige Hessenberg-Matrix mit bulge erstellt und deren Eigenwerte angezeigt. Im zweiten Block wird dann die Prozedur `bulge_chasing` angewandt und die Ergebnismatrix sowie deren Eigenwerte ausgegeben. Hat die Ergebnismatrix die gewünschte Hessenburg-Struktur? Und hat sie weiterhin dieselben Eigenwerte wie die Ausgangsmatrix?

In [7]:
# Beispielmatrix
n = 7
H_start = -1 + 2*rnd.rand(n,n)
H_start = np.triu(H_start,-1)
H_start[2,0] = rnd.random(); H_start[3,0] = rnd.random(); H_start[3,1] = rnd.random()
printmatrix(H_start)

print('Eigenwerte vorher:')
printvector(spla.eigvals(H_start))

    0.283    0.130   -0.629    0.040   -0.867    0.793   -0.552
    0.995    0.663   -0.653    0.206    0.335    0.098    0.657
    0.657    0.469   -0.477    0.153   -0.002   -0.159   -0.616
    0.651    0.002    0.088   -0.309    0.585    0.611   -0.833
    0.000    0.000    0.000    0.832   -0.032    0.219   -0.315
    0.000    0.000    0.000    0.000   -0.288   -0.045    0.384
    0.000    0.000    0.000    0.000    0.000    0.611   -0.364

Eigenwerte vorher:
    -1.131+0.000j     0.530+0.706j     0.530-0.706j    -0.719+0.000j     0.425+0.000j     0.043+0.419j     0.043-0.419j



In [8]:
H_new = bulge_chasing(H_start.copy())
printmatrix(H_new)

print('Eigenwerte nachher:')
printvector(spla.eigvals(H_new))

    0.283    0.190    0.825    0.482    0.277    0.842   -0.604
   -1.359    0.237    0.452    0.360    0.045   -0.874    0.457
    0.000    0.469   -0.616    0.520    0.350   -0.102    0.155
    0.000    0.000    0.356   -0.267    0.082    0.236    0.629
    0.000    0.000    0.000   -0.576   -0.093    0.064    0.801
    0.000    0.000    0.000    0.000   -0.817    0.198    0.769
    0.000    0.000    0.000    0.000    0.000    0.312   -0.023

Eigenwerte nachher:
     0.530+0.706j     0.530-0.706j     0.425+0.000j     0.043+0.419j     0.043-0.419j    -1.131+0.000j    -0.719+0.000j



# Schritt 2: Lemma 6.28 in Action

Nehmen wir an, dass wir nach einer gewissen Anzahl $k$ an Schritten des QR-Algorithmus eine *reellwertige* Hessenberg-Matrix $H_k$ erhalten haben. Dann schauen wir uns zwei weitere Schritte des QR-Algorithmus an, wobei die Shifts $\mu_k$ und $\mu_{k+1}$ entweder beide reell seien, oder ein komplex konjugiertes Paar bilden (also $\mu_{k+1} = \overline{\mu_k}$):
$$
\begin{align*}
H_k - \mu_k I &= Q_k R_k && \text{(QR-Zerlegung)}
\\
H_{k+1} &= R_k Q_k + \mu_k I
\\
H_{k+1} - \mu_{k+1} I &= Q_{k+1} R_{k+1} && \text{(QR-Zerlegung)}
\\
H_{k+2} &= R_{k+1} Q_{k+1} + \mu_{k+1} I
\end{align*}
$$
In der Vorlesung wurde diskutiert, dass 
* $H_{k+2} = Q^H H_k Q$ mit $Q = Q_k Q_{k+1}$ gilt,
* $Q$ und $H_{k+2}$ wieder reelle Matrizen sind, wenn die QR-Zerlegungen so gewählt werden, dass die Diagonalelemente von $R_k$ und $R_{k+1}$ reell sind,
* die Matrix $Q$ zu einer reellwertigen QR-Zerlegung der reellwertigen Matrix $M = H_k^2 - (\mu_k+\mu_{k+1}) H_k + \mu_k \mu_{k+1} I$ gehört (siehe auch Hausaufgabe 11). 

Die Zwischenergebnisse, also die Matrizen $Q_k,Q_{k+1},R_k,R_{k+1}$ und $H_{k+1}$ werden aber im Falle komplexer Shifts dennoch komplexe Einträge haben, sodass in komplexer Arithmetik gerechnet werden muss.

Daher wurde eine zweite Variante diskutiert, mit der auf ganz anderem Wege dasselbe Ergebnis (bis auf Vorzeichen) erzielt wird:
* Wende diejenige (reelle) Householder-Transformation $T_0$, welche die erste Spalte von $M$ auf ein Vielfaches von $e_1$ spiegelt, von links und rechts auf $H_k$ an,
* Transformiere die resultierende Matrix durch bulge chasing (also durch Anwendung geeigneter, reeller Householder-Transformationen $T_1,...,T_{n-2}$) auf Hessenberg-Form. Erhalte Matrix $\widehat{H} = T^\top H_k T$ mit $T = T_0 T_1 \cdots T_{n-2}$.
* Die Matrix $T$ hat (evtl. bis auf das Vorzeichen) dieselbe erste Spalte wie $Q$. Nach Lemma 6.28 stimmen daher sowohl $\widehat{H}$ und $H_{k+2}$ (bis auf Vorzeichen) als auch $T$ und $Q$ (bis auf unterschiedliche Vorzeichen der Spalten) überein. Insbesondere stammt auch $T$ aus einer reellwertigen QR-Zerlegung von $M$, sodass man im QR-Algorithmus genausogut mit der Matrix $\widehat{H} = T^\top H_k T$ statt mit $H_{k+2} = Q^H H_k Q = Q^\top H_k Q$ weiter machen kann.

Der entscheidende Vorteile der zweiten Vorgehensweise ist, dass hier (auch bei komplex konjugierten Shifts) nur reelle Rechenoperationen nötig sind!

Die Matrizen $\widehat{H}$ bzw. $H_{k+2}$ sind übrigens die einzigen Größen, die man für die weiteren Schritte des QR-Algorithmus braucht. Die Matrizen $T$ bzw. $Q$ braucht man sich nicht merken.

**(b) Schreiben Sie eine Prozedur `double_step`, welche die zweite Vorgehensweise umsetzt, d.h. welche**
* **eine Hessenberg-Matrix $H$ und zwei Shifts $\mu_1$,$\mu_2$ (entweder beide reell oder ein komplex konjugiertes Paar) als Eingabe erhält,**
* **die erste Spalte der Matrix $M = H^2 - (\mu_1 + \mu_2) H + \mu_1\mu_2 I$ berechnet (ohne Matrixprodukte!),**
* **die Householder-Transformation, die die erste Spalte von $M$ auf ein Vielfaches von $e_1$ spiegelt, von links und rechts auf $H$ anwendet,**
* **das Ergebnis mittels der Prozedur `bulge_chasing` wieder auf eine Hessenberg-Matrix $\widehat{H}$ transformiert,**
* **und schließlich die Matrix $\widehat{H}$ zurück gibt.**

Hinweise:
* Für beliebige $\mu_1,\mu_2\in\mathbb{C}$ sind natürlich auch $\mu_1+\mu_2, \mu_1\mu_2 \in \mathbb{C}$, sodass auch `numpy` für das Ergebnis komplexe Zahlen als Datentyp verwendet. Wir lassen ja aber nur komplex konjugierte Shifts $\mu_1 = \overline{\mu_2}$ zu, für die $\mu_1+\mu_2 = 2 \mathrm{real}(\mu_1)$ und $\mu_1\mu_2 = |\mu_1|^2$ reell sind. Wenden Sie daher nach Berechnung von $\mu_1+\mu_2$ und $\mu_1\mu_2$ jeweils den Befehl `np.real(...)` an, um den Imaginärteil (der eh Null ist) weg zu lassen und wieder eine Variable mit reellem Datentyp zu erhalten.
* Geeigneter Code zum Testen der Prozedur steht unten bereit.

In [9]:
def double_step(H,mu1,mu2):
    mu_sum = np.real(mu1 + mu2)
    mu_prod = np.real(mu1 * mu2)

    # Berechne erste Spalte von M:
    H_1 = H[0:2,0] # Erste Spalte von H (außer Null-Einträge)
    x = H[0:3,0:2] @ H_1 - mu_sum * H[0:3,0]
    x[0] += mu_prod 
    
    # HH-Vektor der reellen HH-Trafo von x auf alpha*e_1:
    alpha = - np.sign(x[0]) * np.linalg.norm(x)
    v = x
    v[0] -= alpha
    v = v/np.linalg.norm(v)
    
    # Wende Q = I - 2vv^H von links an --> erste drei Zeilen von H bearbeiten (enthalten in allen Spalten Einträge)
    H[0:3,:] += np.outer( (-2*v) , v @ H[0:3,:] )
    
    # Wende Q = I - 2vv^H von rechts an --> erste drei Spalten von H bearbeiten (enthalten nur in den ersten vier Zeilen Einträge)
    H[0:4,0:3] -= np.outer( H[0:4,0:3] @ (2*v) , v )
    
    # Bulge Chasing
    H = bulge_chasing(H)

    return H


Mit dem folgenden Code können Sie Ihre Prozedur wieder testen. Darin wird zunächst eine zufällige Hessenberg-Matrix $H$ erstellt sowie zwei beliebige reelle Shifts oder ein komplex konugiertes Shift-Paar $\mu_1,\mu_2$ gewählt. Dann wird einerseits das Ergebnis der Prozedur `double_step` ausgegeben, und andererseits das Ergebnis, wenn auf klassische Art zwei QR-Algorithmus-Schritte mit Shifts $\mu_1,\mu_2$ durchgeführt werden (beachten Sie, dass der `qr` Befehl aus `scipy` tatsächlich die QR-Zerlegungen komplexer Matrizen so wählt, dass die Diagonaleinträge von $R$ reell sind). Nach der Diskussion oben/aus der Vorlesung sollten beide Ergebnisse bis auf Vorzeichen übereinstimmen.

In [10]:
# Zufällige Hessenberg-Matrix mit Einträgen in [-1,1]:
n = 7
H_start = -1 + 2*rnd.rand(n,n)
H_start = np.triu(H_start,-1)

# Shifts:
mu1 = 2+3j; mu2 = 2-3j # komplex konjugierte Shifts
# mu1 = 2; mu2 = 3 # reelle Shifts

# Zwei klassische QR-Schritte:
H1 = H_start.copy()
Q,R = spla.qr(H1-mu1*np.eye(n))
H1 = R@Q + mu1*np.eye(n)
Q,R = spla.qr(H1-mu2*np.eye(n))
H1 = R@Q + mu2*np.eye(n)
print('Ergebnis nach zwei klassischen QR-Schritten:')
printmatrix(H1)

# Prozedur double_step:
H2 = H_start.copy()
H2 = double_step(H2,mu1,mu2)
print('Ergebnis nach Francis QR-Schritt:')
printmatrix(H2)

Ergebnis nach zwei klassischen QR-Schritten:
     -0.493+0.000j      0.667-0.000j     -1.245+0.000j     -0.124+0.000j     -0.885+0.000j     -0.044+0.000j     -0.834-0.000j
      0.554+0.000j      0.137+0.000j      0.159-0.000j      0.639+0.000j     -0.610-0.000j      0.830+0.000j      0.871+0.000j
      0.000+0.000j     -0.718+0.000j     -0.918+0.000j     -0.482-0.000j      0.231+0.000j      0.303-0.000j      0.028-0.000j
      0.000+0.000j      0.000+0.000j     -0.609-0.000j      0.194+0.000j     -0.072+0.000j     -0.306+0.000j     -0.039+0.000j
      0.000+0.000j      0.000+0.000j      0.000+0.000j      1.245+0.000j     -1.114-0.000j      0.275-0.000j      0.318+0.000j
      0.000+0.000j      0.000+0.000j      0.000+0.000j      0.000+0.000j      0.700+0.000j      0.074-0.000j     -0.658-0.000j
      0.000+0.000j      0.000+0.000j      0.000+0.000j      0.000+0.000j      0.000+0.000j      0.266+0.000j     -0.138+0.000j

Ergebnis nach Francis QR-Schritt:
   -0.493    0.667   -1.245   -

Die Ergebnisse stimmen bis auf manche Vorzeichen überein. Auch wenn im Ergebnis der klassischen QR-Schritte alle Imaginärteile Null sind, erkennt man hier schön, dass zwischendurch in komplexer Arithmetik gerechnet wurde!

# Schritt 3: QR-Algorithmus für reellwertige Matrizen

Nun wollen wir eine Prozedur schreiben, die den QR-Algorithmus auf reellwertige Matrizen anwendet. Im Wesentlichen werden dafür einfach iterativ immer wieder zwei QR-Algorithmus-Schritte mittels der Prozedur `double_step` durchgeführt. Zwei Fragen sind allerdings noch offen:
* Wie wählen wir jeweils die Shifts $\mu_1$ und $\mu_2$?
* Wann brechen wir den Algorithmus ab (z.B. in dem Sinne, dass wir das Problem auf eine kleinere Matrix reduzieren können)?

Dazu fahren wir eine ähnliche Strategie wie beim QR-Algorithmus für komplexwertige Matrizen aus Programmieraufgabe 3, arbeiten allerdings mit dem $2\times2$-Block $\widetilde{H}$ rechts unten in der Matrix $H$, also mit
$$
    \widetilde{H} = \begin{pmatrix} h_{n-1,n-1} & h_{n-1,n} \\ h_{n,n-1} & h_{n,n} \end{pmatrix}
$$
anstatt nur mit dem letzten Element $h_{n,n}$. In jedem Schritt bestimmen wir die beiden Eigenwerte dieses $2\times2$-Blocks analytisch und verwenden diese als Shifts. Da auch $\widetilde{H}$ reelle Einträge hat, sind die beiden Eigenwerte, und somit die Shifts, wie gefordert entweder beide reell oder ein komplex konjugiertes Paar. 

Die Hoffnung ist, dass wir mit diesen beiden Shifts zwei Eigenwerte der Matrix $H$ nach wenigen Schritten gut approximieren und somit den $2\times2$-Block am Ende abkopppeln können. Dazu müssen wir das Element $h_{n-1,n-2}$ beobachten. Wenn dieses Element (quasi) Null ist, ist die Matrix reduzibel. Die Eigenwerte des $2\times2$ Blocks können wir leicht berechnen (machen wir ja auch ständig für die Shifts). Die Eigenwerte der verbleibenden $(n-2)\times(n-2)$-Matrix berechnen wir, indem wir die Prozedur rekursiv aufrufen. Konkret teilen wir die Matrix auf, wenn
$$ 
|h_{n-1,n-2}| \leq \texttt{eps} \left( |h_{n-2,n-2,}| + |h_{n-1,n-1}| \right)
$$
mit der Maschinengenauigkeit $\texttt{eps}$ gilt.

Zusätzlich behalten wir auch das alte Abbruchkriterium
$$
|h_{n,n-1}| \leq \texttt{eps} \left( |h_{n-1,n-1,}| + |h_{n,n}| \right)
$$
bei, falls eine Abkopplung nur des letzten Elements stattfindet.

**(c) Ändern Sie die Prozedur `qr_alg_shift` aus Programmieraufgabe 3 (h) so zu einer Prozedur `qr_alg_real` ab, dass sie die oben beschriebene Strategie umsetzt. Dazu sollten Sie insbesondere folgendes ändern:**
* **Ist die eingegebene Matrix $H$ eine $2\times2$-Matrix, dann sollten die Prozedur (wie auch bisher schon im $1\times1$-Fall) einfach die beiden Eigenwerte von $H$ zurück geben.**
* **Für $n\not\in\{1,2\}$ müssen in jedem Schritt zunächst die beiden Shifts berechnet werden. Um die eigentlichen zwei QR-Schritte durchzuführen, muss dann nur noch die Prozedur `double_step` angewandt werden.**
* **Das neue Abbruchkritierum sowie sinnvolle Anweisungen für den Fall des Abbruchs müssen ergänzt werden. Beachten Sie beim rekursiven Aufruf der Prozedur (insbesondere im alten Abbruchkriterium), dass Sie den richtigen, neuen Namen der Prozedur verwenden.**

Zur analytischen Berechnung von Eigenwerten reeller $2\times2$-Matrizen können Sie die bereitgestellte Prozedur `ew_2x2` verwenden. Diese liefert einen Vektor aus beiden Eigenwerten zurück (und zwar vom Datentyp "reelle Zahlen", falls die Eigenwerte reell sind, und vom Typ "komplexe Zahl", falls es sich um ein komplex konjugiertes Paar handelt).

In [11]:
def ew_2x2(A):
    det_A = A[0,0]*A[1,1] - A[1,0]*A[0,1]
    spur_A = A[0,0] + A[1,1]
    diskriminante = spur_A**2 - 4*det_A
    if diskriminante < 0:
        lam = ( spur_A + np.array([1j,-1j]) * np.sqrt(-diskriminante) )/2
    else:
        lam = ( spur_A + np.array([1,-1]) * np.sqrt(diskriminante) )/2

    return lam

In [12]:
def qr_alg_real(H,kMax,printShifts):
    ep = np.finfo(np.float64).eps # Maschinengenauigkeit
    n = np.size(H,0)
    if n == 1:
        lam = H[0,0]
        print('Übriger EW:',lam)
        return lam
    elif n == 2:
        lam = ew_2x2(H[-2:,-2:])
        print('Übrige EWe:',lam[0],',',lam[1])
        return lam
    else:
        for k in range(kMax):            
            # Bestimme beide Shifts aus 2x2 Block
            mu1,mu2 = ew_2x2(H[-2:,-2:])

            # Ggf.: Ausgabe aktueller Shifts
            if printShifts: 
                print('Schritt',k,', Shifts:')
                print(mu1,',',mu2)

            # Francis' QR-Schritt
            H = double_step(H,mu1,mu2)

            # Ggf. Ausgabe der aktuellen Iterierten
            if printShifts:
                print('Schritt',k,', Nächste Iterierte:')
                printmatrix(H)

            # Abbruchkritierium: 2x2-Block abgekoppelt?
            if np.abs(H[-2,-3]) <= ep * (np.abs(H[-2,-2]) + np.abs(H[-3,-3])):
                mu1,mu2 = ew_2x2(H[-2:,-2:])
                print('Deflation in Schritt',k,'. Zwei EWe ',mu1,',',mu2,'. Restmatrix',n-2,'x',n-2,'.')
                lam = qr_alg_real(H[0:-2,0:-2],kMax,printShifts)
                lam = np.append(lam,mu1)
                lam = np.append(lam,mu2)
                return lam
                
            # Abbruchkritierium: 1x1-Block abgekoppelt?
            if np.abs(H[-1,-2]) <= ep * (np.abs(H[-1,-1]) + np.abs(H[-2,-2])):
                mu = H[-1,-1]
                print('Deflation in Schritt',k,'. Ein EW ',mu,'. Restmatrix',n-1,'x',n-1,'.')
                lam = qr_alg_real(H[0:-1,0:-1],kMax,printShifts)
                lam = np.append(lam,mu)
                return lam
                
        # Abbruch, falls kMax erreicht wurde:
        print('Abbruch: kMax erreicht! Restmatrix:')
        printmatrix(H)
        return []

**(d) Führen Sie den folgenden Code-Block aus, um Ihre Prozedur auf die (in Hessenberg-Form gebrachte) Matrix $A$ von ganz oben anzuwenden und somit die Funktionalität Ihres Codes zu überprüfen.**

In [13]:
lam = qr_alg_real(np.copy(A_hess),20,True)
print('Berechnete Eigenwerte:')
printvector(lam)

Schritt 0 , Shifts:
1.391437032467587 , 0.3087260374830795
Schritt 0 , Nächste Iterierte:
    1.694    2.274    4.372   36.711    0.533
    0.044    0.514   -0.801   -0.158   -1.549
    0.000    1.491    1.594    0.754   -0.162
    0.000    0.000    0.111    0.733    0.319
    0.000    0.000    0.000    0.200    0.965

Schritt 1 , Shifts:
1.1272760636323278 , 0.5709991966496115
Schritt 1 , Nächste Iterierte:
    2.146   -2.102    4.030   26.507  -25.328
   -0.077    0.643   -0.868    0.070    2.137
    0.000    1.620    1.201   -1.545    1.601
    0.000    0.000   -0.004    0.600    0.221
    0.000    0.000    0.000    0.174    0.910

Schritt 2 , Shifts:
1.0046782991352745 , 0.5048802457787792
Schritt 2 , Nächste Iterierte:
    2.001   -3.962    2.229   36.742   -1.357
    0.051    0.460   -1.363    0.225   -1.269
    0.000    0.799    1.538    0.169   -0.987
    0.000    0.000    0.000    0.561    0.144
    0.000    0.000    0.000    0.187    0.939

Schritt 3 , Shifts:
1.0000095948501

Wie in Programmieraufgabe 3 können wir mit dieser Prozedur jetzt auch die Eigenwerte beliebiger anderer reellwertiger Matrizen (egal ob mit komplexen Eigenwertpaaren oder nicht) berechnen. Hier zum Beispiel für eine zufällige Matrix: 

In [17]:
n = 15
B = -1 + 2*rnd.rand(n,n)
B_hess = hess(B)

In [18]:
print('Eigenwerte laut Python:')
printvector(spla.eigvals(B_hess))

print('Francis QR-Algorithmus:')
lam = qr_alg_real(B_hess.copy(),100,False)
print('\nEigenwerte:')
printvector(lam)

Eigenwerte laut Python:
    -2.292+0.000j    -0.519+1.710j    -0.519-1.710j    -1.387+0.227j    -1.387-0.227j     1.866+0.000j     1.688+0.648j     1.688-0.648j     0.320+1.487j     0.320-1.487j    -0.047+0.746j    -0.047-0.746j    -0.045+0.000j     1.025+0.000j     0.573+0.000j

Francis QR-Algorithmus:
Deflation in Schritt 3 . Zwei EWe  1.0251017668560372 , 0.5731133342262478 . Restmatrix 13 x 13 .
Deflation in Schritt 3 . Ein EW  -0.0454274278500217 . Restmatrix 12 x 12 .
Deflation in Schritt 2 . Zwei EWe  (-0.04711068997978545+0.7455011162344969j) , (-0.04711068997978545-0.7455011162344969j) . Restmatrix 10 x 10 .
Deflation in Schritt 4 . Zwei EWe  (0.31991189989659874+1.4871994378709386j) , (0.31991189989659874-1.4871994378709386j) . Restmatrix 8 x 8 .
Deflation in Schritt 6 . Ein EW  1.865877380615021 . Restmatrix 7 x 7 .
Deflation in Schritt 2 . Zwei EWe  (-1.38696111163798+0.2271194642442706j) , (-1.38696111163798-0.2271194642442706j) . Restmatrix 5 x 5 .
Deflation in Schritt 2 